# SCiO USB Hardware Recovery Notebook

This notebook is for **read-first hardware investigation**. Its goal is to recover device metadata, firmware parameter checksums, file inventories, and any evidence of undocumented read paths that could help reconstruct the raw-to-331-band transform offline.

It is intentionally separate from `01_scio_usb.ipynb`:

- `01_scio_usb.ipynb`: existing USB/capture notebook. Do not modify it for this investigation.
- `01b_scio_usb.ipynb`: hardware recovery and reverse-engineering notebook.

No cells in this notebook should write to the device by default.

## Safety Rules

Default behavior is read-only.

Do not run write/state-changing commands unless you have deliberately edited a cell and opted in. In particular:

- `FILE_DOWNLOAD` (`0x81`) appears to be host-to-device firmware write. It is **not** proven to read files back.
- `PARAMETER_SET` (`0x07`) writes device parameters.
- `READY_FOR_WR` (`0x0E`) and `CLEAR_READY_FOR_WR` (`0x11`) change device state.
- `RESET_DEVICE` (`0x83`) may reboot/disconnect the device.

The most valuable first-pass hardware data is:

1. device/BLE identifiers and I2S tag/config strings;
2. firmware file inventory;
3. file headers/checksums for IDs `100-103`;
4. raw responses from strictly read-only experimental probes.

`READ_FILE_HEADER` likely returns checksums/headers only. It probably does **not** return the full parameter file body.

## Command Framing

Android-derived SCiO serial/BLE commands are framed as:

```text
[seq, 0xBA, cmd, payload_len_le16, payload...]
```

- `seq`: one-byte sequence counter.
- `0xBA`: protocol marker. In Java signed-byte form this appears as `-70`.
- `cmd`: one-byte command ID. Java signed bytes must be converted to unsigned bytes before writing.
- `payload_len_le16`: little-endian unsigned 16-bit payload length.
- `payload`: command-specific bytes.

This notebook logs raw command and response bytes so failed hypotheses remain useful.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
import base64
import json
import struct
import time

try:
    import serial
    import serial.tools.list_ports
except ImportError:
    serial = None

OUTPUT_DIR = Path("hardware_recovery_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

RECOVERED_PARAM_DIR = Path("recovered_firmware_params")

SCIO_USB_VID_PID = "0451:16AA"
SCIO_MARKER = 0xBA
DEFAULT_BAUDRATE = 115200
DEFAULT_TIMEOUT_S = 2.0

FIRMWARE_PARAM_IDS = {
    "deadPixelsIndices": 100,
    "centers": 101,
    "bins": 102,
    "nPixelsPerBin": 103,
}

COMMANDS = {
    "READ_DEVICE_STATUS": 0x00,
    "READ_DEVICE_ID": 0x01,
    "SAMPLE_SPECTRUM": 0x02,
    "READ_TEMPERATURE": 0x03,
    "READ_BATTERY_STATE": 0x05,
    "READ_EVENT_LOG": 0x06,
    "PARAMETER_SET": 0x07,
    "PARAMETER_GET": 0x08,
    "READY_FOR_WR": 0x0E,
    "CLEAR_READY_FOR_WR": 0x11,
    "FILE_DOWNLOAD": 0x81,
    "RESET_DEVICE": 0x83,
    "READ_BLE_ID": 0x84,
    "READ_BLE_STATUS": 0x85,
    "READ_FILE_HEADER": 0x87,
    "WRITE_BLE": 0x9A,
    "READ_BLE": 0x9B,
    "WRITE_USER_DEVICE_NAME": 0x91,
    "READ_FILE_LIST": 0x94,
}

READ_ONLY_COMMANDS = {
    "READ_DEVICE_STATUS",
    "READ_DEVICE_ID",
    "READ_TEMPERATURE",
    "READ_BATTERY_STATE",
    "READ_EVENT_LOG",
    "PARAMETER_GET",
    "READ_BLE_ID",
    "READ_BLE_STATUS",
    "READ_BLE",
    "READ_FILE_HEADER",
    "READ_FILE_LIST",
}

## Serial Discovery

This only lists ports. It does not open the device or send commands.

In [ ]:
def list_serial_ports():
    if serial is None:
        raise RuntimeError("pyserial is not installed in this environment.")
    rows = []
    for port in serial.tools.list_ports.comports():
        rows.append({
            "device": port.device,
            "description": port.description,
            "hwid": port.hwid,
            "vid": None if port.vid is None else f"{port.vid:04X}",
            "pid": None if port.pid is None else f"{port.pid:04X}",
            "is_likely_scio": (
                port.vid is not None
                and port.pid is not None
                and f"{port.vid:04X}:{port.pid:04X}".upper() == SCIO_USB_VID_PID
            ),
        })
    return rows


# Safe to run: lists ports only.
list_serial_ports()

## Command And Response Helpers

The functions below define command framing, response capture, parsing helpers, and JSON logging. They do not talk to hardware until `SCiORecoverySession.send_read_command(...)` is called.

In [ ]:
def unsigned_byte(value):
    return int(value) & 0xFF


def create_command(cmd, payload=b"", seq=1):
    payload = bytes(payload)
    return bytes([
        unsigned_byte(seq),
        SCIO_MARKER,
        unsigned_byte(cmd),
    ]) + struct.pack("<H", len(payload)) + payload


def now_iso():
    return datetime.now(timezone.utc).isoformat()


def b64(data):
    return base64.b64encode(bytes(data)).decode("ascii")


def hexstr(data):
    return bytes(data).hex()


def parse_u32le_words(data):
    data = bytes(data)
    usable = len(data) - (len(data) % 4)
    return list(struct.unpack("<" + "I" * (usable // 4), data[:usable])) if usable else []


def parse_ascii_guess(data):
    data = bytes(data)
    chars = []
    for byte in data:
        if 32 <= byte <= 126:
            chars.append(chr(byte))
        elif byte in (9, 10, 13):
            chars.append(" ")
        else:
            chars.append(".")
    return "".join(chars)


def parse_file_list_entries(data):
    data = bytes(data)
    entries = []
    for offset in range(0, len(data) - 7, 8):
        file_type, file_version = struct.unpack("<II", data[offset:offset + 8])
        entries.append({
            "offset": offset,
            "file_type": file_type,
            "file_version": file_version,
            "known_name": next((name for name, fid in FIRMWARE_PARAM_IDS.items() if fid == file_type), None),
        })
    return entries


def parse_file_header(data):
    words = parse_u32le_words(data)
    parsed = {"u32le_words": words}
    if len(words) >= 4:
        parsed["checksum_word3"] = words[3]
    return parsed


def save_json_artifact(prefix, obj, output_dir=OUTPUT_DIR):
    output_dir.mkdir(exist_ok=True)
    safe_prefix = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in prefix)
    path = output_dir / f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_{safe_prefix}.json"
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    return path


@dataclass
class CommandResult:
    timestamp_utc: str
    command_name: str
    command_byte: int
    payload_hex: str
    request_hex: str
    response_hex: str
    response_b64: str
    response_len: int
    notes: str = ""
    parsed: dict | None = None

## Hardware Session

Instantiate this only when the SCiO is connected. Sending commands is not automatic.

The session guards command names against accidental writes. To probe unsupported write-like commands, you would need to edit the code intentionally; there is no write helper in this notebook.

In [ ]:
class SCiORecoverySession:
    def __init__(self, port, baudrate=DEFAULT_BAUDRATE, timeout=DEFAULT_TIMEOUT_S):
        if serial is None:
            raise RuntimeError("pyserial is not installed in this environment.")
        self.port = port
        self.baudrate = baudrate
        self.timeout = timeout
        self.seq = 1
        self.ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)

    def close(self):
        if getattr(self, "ser", None) is not None:
            self.ser.close()

    def next_seq(self):
        value = self.seq
        self.seq = 1 + (self.seq % 255)
        return value

    def read_available_response(self, settle_s=0.15, max_wait_s=DEFAULT_TIMEOUT_S):
        deadline = time.time() + max_wait_s
        chunks = []
        while time.time() < deadline:
            waiting = self.ser.in_waiting
            if waiting:
                chunks.append(self.ser.read(waiting))
                time.sleep(settle_s)
                continue
            if chunks:
                break
            time.sleep(0.02)
        return b"".join(chunks)

    def send_read_command(self, command_name, payload=b"", notes="", parser=None):
        if command_name not in READ_ONLY_COMMANDS:
            raise RuntimeError(f"{command_name} is not in READ_ONLY_COMMANDS; refusing to send it from this notebook.")
        cmd = COMMANDS[command_name]
        packet = create_command(cmd, payload=payload, seq=self.next_seq())
        self.ser.write(packet)
        self.ser.flush()
        response = self.read_available_response()
        parsed = parser(response) if parser is not None else None
        result = CommandResult(
            timestamp_utc=now_iso(),
            command_name=command_name,
            command_byte=cmd,
            payload_hex=hexstr(payload),
            request_hex=hexstr(packet),
            response_hex=hexstr(response),
            response_b64=b64(response),
            response_len=len(response),
            notes=notes,
            parsed=parsed,
        )
        path = save_json_artifact(command_name, asdict(result))
        return result, path

    def read_device_status(self):
        return self.send_read_command("READ_DEVICE_STATUS", notes="Read-only device status.")

    def read_device_id(self):
        return self.send_read_command("READ_DEVICE_ID", notes="Read-only DSP/Aptina/device identifier probe.", parser=lambda d: {"ascii_guess": parse_ascii_guess(d), "u32le_words": parse_u32le_words(d)})

    def read_ble_id(self):
        return self.send_read_command("READ_BLE_ID", notes="Read-only BLE ID/name/I2S-tag probe.", parser=lambda d: {"ascii_guess": parse_ascii_guess(d), "u32le_words": parse_u32le_words(d)})

    def read_ble_status(self):
        return self.send_read_command("READ_BLE_STATUS", notes="Read-only BLE status probe.", parser=lambda d: {"u32le_words": parse_u32le_words(d)})

    def read_ble(self):
        return self.send_read_command("READ_BLE", notes="Read-only BLE configuration probe.", parser=lambda d: {"ascii_guess": parse_ascii_guess(d), "u32le_words": parse_u32le_words(d)})

    def read_battery(self):
        return self.send_read_command("READ_BATTERY_STATE", notes="Read-only battery state.", parser=lambda d: {"u32le_words": parse_u32le_words(d)})

    def read_temperature(self):
        return self.send_read_command("READ_TEMPERATURE", notes="Read-only temperature.", parser=lambda d: {"u32le_words": parse_u32le_words(d)})

    def read_event_log(self):
        return self.send_read_command("READ_EVENT_LOG", notes="Read-only event log probe.", parser=lambda d: {"ascii_guess": parse_ascii_guess(d), "u32le_words": parse_u32le_words(d)})

    def read_file_list(self):
        return self.send_read_command("READ_FILE_LIST", notes="Read-only firmware file inventory.", parser=lambda d: {"entries": parse_file_list_entries(d), "u32le_words": parse_u32le_words(d)})

    def read_file_header(self, file_id):
        payload = struct.pack("<I", int(file_id))
        return self.send_read_command("READ_FILE_HEADER", payload=payload, notes=f"Read-only header/checksum for file_id={file_id}.", parser=parse_file_header)

    def parameter_get(self, parameter_id, payload_shape="<I"):
        payload = struct.pack(payload_shape, int(parameter_id))
        return self.send_read_command("PARAMETER_GET", payload=payload, notes=f"Experimental read-only PARAMETER_GET id={parameter_id} shape={payload_shape}.", parser=lambda d: {"ascii_guess": parse_ascii_guess(d), "u32le_words": parse_u32le_words(d)})

## Recommended Read-Only Interrogation Order

Uncomment and run this section only with the hardware connected.

Recommended order:

1. list ports and choose the SCiO port;
2. read device/BLE identifiers;
3. read battery/temperature/status;
4. read file list;
5. read headers for firmware parameter IDs `100-103`;
6. read headers for every file ID reported by file list;
7. only then run bounded experimental `PARAMETER_GET` probes.

In [ ]:
# HARDWARE CELL: intentionally disabled by default.
# Uncomment and edit SCIO_PORT after connecting hardware.
#
# SCIO_PORT = "COM3"
# session = SCiORecoverySession(SCIO_PORT)
# try:
#     results = {}
#     for name, func in [
#         ("device_status", session.read_device_status),
#         ("device_id", session.read_device_id),
#         ("ble_id", session.read_ble_id),
#         ("ble_status", session.read_ble_status),
#         ("ble_config", session.read_ble),
#         ("battery", session.read_battery),
#         ("temperature", session.read_temperature),
#     ]:
#         result, path = func()
#         results[name] = {"artifact": str(path), "response_len": result.response_len, "parsed": result.parsed}
#     save_json_artifact("read_only_identity_summary", results)
# finally:
#     session.close()

## Firmware File Inventory And Checksums

This is the most important hardware recovery section for offline decoding. The four target firmware parameter files are:

- `deadPixelsIndices` (`100`)
- `centers` (`101`)
- `bins` (`102`)
- `nPixelsPerBin` (`103`)

The expected outcome is probably checksum/header data, not the parameter body. Preserve the JSON artifacts anyway; checksums can later match recovered blobs from old app preferences or backups.

In [ ]:
def summarize_file_header_result(result):
    parsed = result.parsed or {}
    words = parsed.get("u32le_words", [])
    return {
        "response_len": result.response_len,
        "u32le_words": words,
        "checksum_word3": parsed.get("checksum_word3"),
        "response_hex": result.response_hex,
        "response_b64": result.response_b64,
    }


# HARDWARE CELL: intentionally disabled by default.
# Uncomment after creating a session.
#
# session = SCiORecoverySession(SCIO_PORT)
# try:
#     firmware_headers = {}
#     for name, file_id in FIRMWARE_PARAM_IDS.items():
#         result, path = session.read_file_header(file_id)
#         firmware_headers[name] = {
#             "file_id": file_id,
#             "artifact": str(path),
#             **summarize_file_header_result(result),
#         }
#     save_json_artifact("firmware_param_headers_100_103", firmware_headers)
# finally:
#     session.close()

## Read Headers For File IDs Reported By File List

If `READ_FILE_LIST` returns entries, this helper reads headers for every listed file type. This may reveal additional IDs or checksum/version relationships.

In [ ]:
# HARDWARE CELL: intentionally disabled by default.
# Uncomment after creating a session.
#
# session = SCiORecoverySession(SCIO_PORT)
# try:
#     file_list_result, file_list_path = session.read_file_list()
#     entries = (file_list_result.parsed or {}).get("entries", [])
#     all_headers = {
#         "file_list_artifact": str(file_list_path),
#         "entries": entries,
#         "headers": {},
#     }
#     for entry in entries:
#         file_id = entry["file_type"]
#         result, path = session.read_file_header(file_id)
#         all_headers["headers"][str(file_id)] = {
#             "artifact": str(path),
#             "file_list_entry": entry,
#             **summarize_file_header_result(result),
#         }
#     save_json_artifact("all_reported_file_headers", all_headers)
# finally:
#     session.close()

## Experimental Read-Only `PARAMETER_GET` Probes

The app defines `PARAMETER_GET` (`0x08`), but the exact payload and response schema are not proven. This section only tries bounded read-only probes.

Suggested first IDs:

- `0-32`
- `89-103`
- any file IDs reported by `READ_FILE_LIST`

If a probe causes instability, stop using it. Preserve the raw artifact JSONs.

In [ ]:
def parameter_probe_plan(ids=None, payload_shapes=("<I", "<H", "<B")):
    if ids is None:
        ids = list(range(0, 33)) + [89, 90, 91, 92, 100, 101, 102, 103]
    for parameter_id in ids:
        for payload_shape in payload_shapes:
            max_value = {">B": 255, "<B": 255, ">H": 65535, "<H": 65535}.get(payload_shape)
            if max_value is not None and int(parameter_id) > max_value:
                continue
            yield int(parameter_id), payload_shape


# HARDWARE CELL: intentionally disabled by default.
# Uncomment after creating a session. Keep ranges small.
#
# session = SCiORecoverySession(SCIO_PORT)
# try:
#     probe_rows = []
#     for parameter_id, payload_shape in parameter_probe_plan():
#         result, path = session.parameter_get(parameter_id, payload_shape=payload_shape)
#         probe_rows.append({
#             "parameter_id": parameter_id,
#             "payload_shape": payload_shape,
#             "artifact": str(path),
#             "response_len": result.response_len,
#             "parsed": result.parsed,
#             "response_hex": result.response_hex,
#         })
#     save_json_artifact("parameter_get_probe_summary", probe_rows)
# finally:
#     session.close()

## Undocumented Read-Path Exploration

This section creates read-only payload variants for commands that are already read commands. It does **not** call `FILE_DOWNLOAD`.

The idea is to preserve evidence for possible undocumented body reads without sending write commands. Run only a few variants at a time.

In [ ]:
def read_file_header_payload_variants(file_id):
    file_id = int(file_id)
    return [
        ("u32_file_id", struct.pack("<I", file_id)),
        ("u16_file_id", struct.pack("<H", file_id)),
        ("u32_file_id_offset0_len0", struct.pack("<III", file_id, 0, 0)),
        ("u32_file_id_offset0_len256", struct.pack("<III", file_id, 0, 256)),
        ("u32_file_id_offset0_len1024", struct.pack("<III", file_id, 0, 1024)),
    ]


# HARDWARE CELL: intentionally disabled by default.
# Uncomment after creating a session. Start with one file ID.
#
# session = SCiORecoverySession(SCIO_PORT)
# try:
#     exploratory = []
#     for label, payload in read_file_header_payload_variants(100):
#         result, path = session.send_read_command(
#             "READ_FILE_HEADER",
#             payload=payload,
#             notes=f"Exploratory READ_FILE_HEADER payload variant {label}; read-only command.",
#             parser=parse_file_header,
#         )
#         exploratory.append({
#             "label": label,
#             "payload_hex": payload.hex(),
#             "artifact": str(path),
#             "response_len": result.response_len,
#             "parsed": result.parsed,
#             "response_hex": result.response_hex,
#         })
#     save_json_artifact("read_file_header_payload_variants_file100", exploratory)
# finally:
#     session.close()

## Compare Device Header Checksums With Recovered Parameter Files

When recovered blobs are found, place them in `notebooks/recovered_firmware_params/`.

This helper computes first-four-byte checksum interpretations so they can be compared against `READ_FILE_HEADER` outputs.

In [ ]:
def maybe_base64_decode(raw):
    stripped = raw.strip()
    try:
        text = stripped.decode("ascii")
    except UnicodeDecodeError:
        return raw
    compact = "".join(text.split())
    if not compact:
        return raw
    try:
        return base64.urlsafe_b64decode(compact + ("=" * (-len(compact) % 4)))
    except Exception:
        return raw


def inspect_recovered_param_files(param_dir=RECOVERED_PARAM_DIR):
    param_dir = Path(param_dir)
    rows = []
    if not param_dir.exists():
        return rows
    for path in sorted(param_dir.iterdir()):
        if path.is_dir():
            continue
        raw_file = path.read_bytes()
        raw = maybe_base64_decode(raw_file)
        row = {
            "path": str(path),
            "file_bytes": len(raw_file),
            "decoded_bytes": len(raw),
            "first4_le": None,
            "first4_be": None,
            "name_hint": path.stem,
        }
        if len(raw) >= 4:
            row["first4_le"] = struct.unpack("<I", raw[:4])[0]
            row["first4_be"] = struct.unpack(">I", raw[:4])[0]
        rows.append(row)
    return rows


recovered_param_inspection = inspect_recovered_param_files()
recovered_param_inspection

## Preservation Checklist

After a hardware session, keep:

- every JSON artifact in `notebooks/hardware_recovery_outputs/`;
- the chosen COM port and OS/device details;
- whether the device had old calibration data;
- all `READ_FILE_LIST` entries;
- all `READ_FILE_HEADER` responses for `100-103`;
- raw response hex/Base64 for experimental probes;
- any recovered firmware blobs placed in `notebooks/recovered_firmware_params/`.

Then update `INFO.md` with new command outcomes before changing the offline decoder.